In [15]:
import os
import openml
import pandas as pd
from tqdm import tqdm
from openml import config
import numpy as np
from collections import defaultdict
import random
import json
from sklearn.preprocessing import LabelEncoder

# Set your OpenML API key
config.apikey = 'c0d6200b271e73a8aec0904980876c3c'

In [2]:
with open("flows/filtered_flow_algorithm_mapping_v2.json", "r") as f:
    flow_map = json.load(f)

valid_flow_ids = set(map(int, flow_map.keys()))

In [3]:
suite_id = 225
benchmark_suite = openml.study.get_suite(suite_id)
task_ids = benchmark_suite.tasks

In [5]:
raw_dir = os.path.join("runs", "raw")
os.makedirs(raw_dir, exist_ok=True)

for task_id in tqdm(task_ids, desc="Downloading runs for tasks"):
    try:
        runs_df = openml.runs.list_runs(task=[task_id], output_format='dataframe')

        if runs_df.empty:
            print(f"⚠️ No runs found for task {task_id}. Skipping.")
            continue

        runs_df.set_index('run_id', inplace=True)
        output_path = os.path.join(raw_dir, f"task_{task_id}_runs.csv")
        runs_df.to_csv(output_path)

    except Exception as e:
        print(f"Error fetching runs for task {task_id}: {e}")

In [6]:
raw_dir = os.path.join("runs", "raw")
filtered_dir = os.path.join("runs", "filtered")
log_dir = os.path.join("log")
log_path = os.path.join(log_dir, "filtering.csv")

os.makedirs(filtered_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

log_rows = []

for filename in tqdm(os.listdir(raw_dir), desc="Filtering runs"):
    if filename.startswith("task_") and filename.endswith("_runs.csv"):
        task_id = filename.split("_")[1]
        input_path = os.path.join(raw_dir, filename)
        output_path = os.path.join(filtered_dir, filename)

        try:
            runs_df = pd.read_csv(input_path)
            if 'flow_id' not in runs_df.columns:
                log_rows.append({
                    'task_id': task_id,
                    'filename': filename,
                    'before_count': len(runs_df),
                    'after_count': None,
                    'reduction': None,
                    'status': "No 'flow_id' column"
                })
                continue

            filtered_runs_df = runs_df[runs_df['flow_id'].isin(valid_flow_ids)]
            filtered_runs_df.to_csv(output_path, index=False)

            before = len(runs_df)
            after = len(filtered_runs_df)
            reduction = (before - after) / before if before > 0 else 0

            log_rows.append({
                'task_id': task_id,
                'filename': filename,
                'before_count': before,
                'after_count': after,
                'reduction': f"{reduction:.2%}",
                'status': "Success"
            })

        except Exception as e:
            log_rows.append({
                'task_id': task_id,
                'filename': filename,
                'before_count': None,
                'after_count': None,
                'reduction': None,
                'status': f"Error: {str(e)}"
            })

log_df = pd.DataFrame(log_rows)
log_df.to_csv(log_path, index=False)
print(f"Filter log saved to {log_path}")

Filtering runs: 100%|██████████| 54/54 [00:05<00:00, 10.02it/s]

Filter log saved to log/filtering.csv


In [7]:
algo_to_flows = defaultdict(list)

for flow_id_str, details in flow_map.items():
    algo_type = details.get("algorithm_type", "Unknown")
    flow_id = int(flow_id_str)  
    algo_to_flows[algo_type].append(flow_id)

In [9]:
filtered_dir = os.path.join("runs", "filtered")
splits_dir = os.path.join("runs", "algorithm_splits")
log_dir = os.path.join("log")
summary_log_path = os.path.join(log_dir, "algorithm_split_summary.csv")

os.makedirs(splits_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# === flow_id to algorithm_type mapping ===
flow_to_algorithm = {
    int(fid): entry["algorithm_type"]
    for fid, entry in flow_map.items()
}

# === Track all algorithms used ===
all_algorithms = set(flow_to_algorithm.values())

# === Prepare summary log as wide-format table ===
summary_data = []

# === Process each filtered run file ===
for filename in tqdm(os.listdir(filtered_dir), desc="Splitting by algorithm"):
    if filename.startswith("task_") and filename.endswith("_runs.csv"):
        task_id = filename.split("_")[1]
        input_path = os.path.join(filtered_dir, filename)

        try:
            runs_df = pd.read_csv(input_path)
            if 'run_id' not in runs_df.columns or 'flow_id' not in runs_df.columns:
                print(f"⚠️ Skipping {filename}: missing required columns.")
                continue

            runs_df.set_index("run_id", inplace=True)
            task_output_dir = os.path.join(splits_dir, f"task_{task_id}")
            os.makedirs(task_output_dir, exist_ok=True)

            # Count runs per algorithm
            alg_counts = {alg: 0 for alg in all_algorithms}
            alg_groups = defaultdict(list)

            for run_id, row in runs_df.iterrows():
                flow_id = row["flow_id"]
                algo_type = flow_to_algorithm.get(flow_id)
                if algo_type:
                    alg_groups[algo_type].append(run_id)
                    alg_counts[algo_type] += 1

            for algo_type, run_ids in alg_groups.items():
                filename_out = f"{algo_type.lower().replace(' ', '_')}_runs.csv"
                output_path = os.path.join(task_output_dir, filename_out)
                runs_df.loc[run_ids].to_csv(output_path)

            # Add row to summary log
            summary_data.append({"task_id": task_id, **alg_counts})

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.set_index("task_id")
summary_df = summary_df.sort_index()
summary_df.to_csv(summary_log_path)
print(f"Algorithm run count summary saved to {summary_log_path}")


Splitting by algorithm: 100%|██████████| 54/54 [00:33<00:00,  1.61it/s]

Algorithm run count summary saved to log/algorithm_split_summary.csv


In [11]:
random.seed(42)

# === Setup paths ===
filtered_dir = os.path.join("runs", "filtered")
splits_dir = os.path.join("runs", "algorithm_splits")
accuracies_dir = os.path.join("runs", "accuracies")

os.makedirs(splits_dir, exist_ok=True)
os.makedirs(accuracies_dir, exist_ok=True)

# === flow_id to algorithm_type mapping ===
flow_to_algorithm = {
    int(fid): entry["algorithm_type"]
    for fid, entry in flow_map.items()
}

# === Process each filtered run file ===
for filename in tqdm(os.listdir(filtered_dir), desc="Splitting by algorithm"):
    if filename.startswith("task_") and filename.endswith("_runs.csv"):
        task_id = filename.split("_")[1]
        input_path = os.path.join(filtered_dir, filename)

        try:
            runs_df = pd.read_csv(input_path)
            if 'run_id' not in runs_df.columns or 'flow_id' not in runs_df.columns:
                continue

            runs_df.set_index("run_id", inplace=True)
            task_output_dir = os.path.join(splits_dir, f"task_{task_id}")
            os.makedirs(task_output_dir, exist_ok=True)

            acc_task_output_dir = os.path.join(accuracies_dir, f"task_{task_id}")
            os.makedirs(acc_task_output_dir, exist_ok=True)

            alg_groups = defaultdict(list)

            for run_id, row in runs_df.iterrows():
                flow_id = row["flow_id"]
                algo_type = flow_to_algorithm.get(flow_id)
                if algo_type:
                    alg_groups[algo_type].append(run_id)

            for algo_type, run_ids in alg_groups.items():
                filename_out = f"{algo_type.lower().replace(' ', '_')}_runs.csv"
                output_path = os.path.join(task_output_dir, filename_out)
                sampled_run_ids = random.sample(run_ids, min(50, len(run_ids)))
                sampled_df = runs_df.loc[sampled_run_ids].copy()

                # Fetch predictive accuracies
                accuracy_map = {}
                BATCH_SIZE = 50

                def chunks(lst, n):
                    for i in range(0, len(lst), n):
                        yield lst[i:i + n]

                for batch in tqdm(list(chunks(sampled_run_ids, BATCH_SIZE)), desc=f"Fetching accuracy for {algo_type}", leave=False):
                    try:
                        evaluations = openml.evaluations.list_evaluations(
                            function='predictive_accuracy',
                            runs=[int(rid) for rid in batch],
                            output_format='dataframe'
                        )
                        if not evaluations.empty:
                            accuracy_map.update(dict(zip(evaluations['run_id'], evaluations['value'])))
                    except Exception as e:
                        print(f"⚠️ Error fetching accuracies for {algo_type}: {e}")

                sampled_df['predictive_accuracy'] = sampled_df.index.map(lambda x: accuracy_map.get(int(x), None))

                # Save split CSV and sampled accuracy CSV
                runs_df.loc[run_ids].to_csv(output_path)
                acc_filename = f"{algo_type.lower().replace(' ', '_')}_accuracies.csv"
                acc_path = os.path.join(acc_task_output_dir, acc_filename)
                sampled_df.to_csv(acc_path)

        except Exception as e:
            print(f"Error processing {filename}: {e}")


Splitting by algorithm: 100%|██████████| 54/54 [01:47<00:00,  1.99s/it]


In [13]:
accuracies_dir = os.path.join("runs", "accuracies")
statistics_dir = os.path.join("runs", "statistics")
os.makedirs(statistics_dir, exist_ok=True)

for task_folder in os.listdir(accuracies_dir):
    if not task_folder.startswith("task_"):
        continue

    task_id = task_folder.split("_")[1]
    task_accuracy_dir = os.path.join(accuracies_dir, task_folder)
    statistics_path = os.path.join(statistics_dir, f"task_{task_id}_statistics.csv")

    summary_rows = []

    for filename in os.listdir(task_accuracy_dir):
        if filename.endswith("_accuracies.csv"):
            csv_path = os.path.join(task_accuracy_dir, filename)
            df = pd.read_csv(csv_path)

            if 'predictive_accuracy' not in df.columns or df['predictive_accuracy'].dropna().empty:
                print(f"⚠️ Skipping {filename} — no predictive accuracy.")
                continue

            accs = df['predictive_accuracy'].dropna()

            algo_name = filename.replace("_accuracies.csv", "").replace("_", " ").title()
            stats = {
                'algorithm': algo_name,
                'count': len(accs),
                'mean': accs.mean(),
                'std': accs.std(),
                'min': accs.min(),
                '25%': accs.quantile(0.25),
                '50% (median)': accs.median(),
                '75%': accs.quantile(0.75),
                'max': accs.max(),
                'top10_median': accs.sort_values(ascending=False).head(10).median()
            }

            summary_rows.append(stats)

    # Save summary CSV if any valid accuracies found
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).set_index("algorithm")
        summary_df.to_csv(statistics_path)


Saved summary for task 10101 → runs/statistics/task_10101_statistics.csv
Saved summary for task 14966 → runs/statistics/task_14966_statistics.csv
Saved summary for task 9980 → runs/statistics/task_9980_statistics.csv
Saved summary for task 3481 → runs/statistics/task_3481_statistics.csv
Saved summary for task 9986 → runs/statistics/task_9986_statistics.csv
Saved summary for task 14969 → runs/statistics/task_14969_statistics.csv
Saved summary for task 9981 → runs/statistics/task_9981_statistics.csv
Saved summary for task 37 → runs/statistics/task_37_statistics.csv
Saved summary for task 3903 → runs/statistics/task_3903_statistics.csv
Saved summary for task 2074 → runs/statistics/task_2074_statistics.csv
Saved summary for task 53 → runs/statistics/task_53_statistics.csv
Saved summary for task 3902 → runs/statistics/task_3902_statistics.csv
Saved summary for task 6 → runs/statistics/task_6_statistics.csv
Saved summary for task 36 → runs/statistics/task_36_statistics.csv
Saved summary for 

In [16]:
accuracies_dir = os.path.join("runs", "accuracies")
final_dir = os.path.join("final")
os.makedirs(final_dir, exist_ok=True)
targets_path = os.path.join(final_dir, "targets.csv")

merged_rows = []

for task_folder in os.listdir(accuracies_dir):
    if not task_folder.startswith("task_"):
        continue

    task_id = task_folder.split("_")[1]
    task_accuracy_dir = os.path.join(accuracies_dir, task_folder)

    task_row = {'task_id': task_id}
    top10_medians = {}

    for filename in os.listdir(task_accuracy_dir):
        if filename.endswith("_accuracies.csv"):
            csv_path = os.path.join(task_accuracy_dir, filename)
            df = pd.read_csv(csv_path)

            if 'predictive_accuracy' not in df.columns or df['predictive_accuracy'].dropna().empty:
                continue

            accs = df['predictive_accuracy'].dropna()
            algo_name = filename.replace("_accuracies.csv", "").replace("_", " ").title()
            top10_median = accs.sort_values(ascending=False).head(10).median()
            top10_medians[algo_name] = top10_median

    if not top10_medians:
        continue

    task_row.update(top10_medians)

    best_algo = max(top10_medians.items(), key=lambda x: x[1])[0]
    task_row['best_algorithm'] = best_algo

    merged_rows.append(task_row)

# === Step 2: Create DataFrame and encode label ===
combined_df = pd.DataFrame(merged_rows)
combined_df = combined_df.set_index("task_id")

# Encode best_algorithm
label_encoder = LabelEncoder()
combined_df["best_algorithm_encoded"] = label_encoder.fit_transform(combined_df["best_algorithm"])

# Save to final/targets.csv
combined_df.to_csv(targets_path)

# === Print label mapping ===
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Final targets saved to: {targets_path}")
print("Encoded 'best_algorithm' mapping:")
for algo, code in label_mapping.items():
    print(f"  {code} → {algo}")

combined_df.head()

Final targets saved to: final/targets.csv
Encoded 'best_algorithm' mapping:
  0 → Decision Tree
  1 → Random Forest
  2 → Support Vector Machine
  3 → Xgboost


,Xgboost,Decision Tree,Support Vector Machine,Random Forest,best_algorithm,best_algorithm_encoded
task_id,,,,,,
10101,0.772727,0.762032,0.770053,0.786096,Random Forest,1
14966,0.797120,0.753533,0.770461,0.806318,Random Forest,1
9980,0.914815,0.914815,0.914815,0.916667,Random Forest,1
3481,0.885918,0.829486,0.971335,0.945043,Support Vector Machine,2
9986,0.956003,0.968512,0.992811,0.993458,Random Forest,1
